# 01 – Yelp Data Exploration

Initial exploration of the Yelp Open Dataset to understand the data distribution,
identify relevant POI categories, and guide preprocessing decisions.

**Prerequisites:** download the Yelp dataset and place files under `data/raw/`.
See `data/README.md` for instructions.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src importable when running from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load businesses

In [ ]:
from src.data.yelp_loader import load_businesses

# Limit rows for quick exploration; remove max_rows for full dataset
businesses = load_businesses(raw_dir='../data/raw', max_rows=50_000)
print(f'Shape: {businesses.shape}')
businesses.head()

## 2. Category distribution

In [ ]:
# Explode the comma-separated categories column
cat_series = (
    businesses['categories']
    .dropna()
    .str.split(', ')
    .explode()
    .str.strip()
)
top_cats = cat_series.value_counts().head(30)

fig, ax = plt.subplots(figsize=(10, 7))
top_cats.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 30 Yelp categories (filtered dataset)')
ax.set_xlabel('Number of businesses')
plt.tight_layout()
plt.show()

## 3. Geographic distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(
    businesses['longitude'], businesses['latitude'],
    s=1, alpha=0.3, c=businesses['stars'], cmap='RdYlGn'
)
ax.set_title('POI geographic distribution (coloured by average rating)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 4. Load reviews & check interaction sparsity

In [ ]:
from src.data.yelp_loader import load_reviews
from src.data.preprocessing import filter_interactions

poi_ids = set(businesses['business_id'])
reviews = load_reviews(raw_dir='../data/raw', business_ids=poi_ids, max_rows=200_000)

interactions = filter_interactions(reviews)

n_users = interactions['user_id'].nunique()
n_items = interactions['business_id'].nunique()
n_interactions = len(interactions)
sparsity = 1 - n_interactions / (n_users * n_items)

print(f'Users: {n_users:,}')
print(f'Items: {n_items:,}')
print(f'Interactions: {n_interactions:,}')
print(f'Sparsity: {sparsity:.4%}')

## 5. Interaction count distribution (long-tail)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

interactions['user_id'].value_counts().hist(bins=50, ax=axes[0], log=True)
axes[0].set_title('User interaction count distribution')
axes[0].set_xlabel('# interactions')
axes[0].set_ylabel('# users (log scale)')

interactions['business_id'].value_counts().hist(bins=50, ax=axes[1], log=True)
axes[1].set_title('Item interaction count distribution')
axes[1].set_xlabel('# interactions')
axes[1].set_ylabel('# items (log scale)')

plt.tight_layout()
plt.show()